<a href="https://colab.research.google.com/github/Fatima-Eman-hub/fatima-eman-flyrank-ml-01/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — Freshness multiplier:** "Refreshing content older than 365
days produced a 3.2x health boost and 57x more impressions in this sample."

**My methodology question:** Where does the "health" label come from — is
it the product's own composite `health_score` (a rule-based metric), or an
independently measured outcome? More importantly, does the validation
design support a causal reading of "produced"? A 57x impression multiplier
is a very large effect size, and it's worth asking whether refreshed pages
were already showing early recovery signals *before* the refresh happened
(reverse causality: pages that were about to recover may be the ones a
team chooses to refresh), rather than the refresh itself causing the gain.
Was refresh timing compared against a matched control group of similarly
aged, non-refreshed content, or only against the same pages' own past
performance? This doesn't mean the finding is wrong — it's a genuinely
useful signal either way — but "produced" is a causal verb, and I'd want
to see a matched-baseline or experimental comparison before treating this
as more than a strong observed association.

**Finding B — AI traffic behaves differently:** "High-AI pages get ~9x
more impressions despite weaker average Google position."

**My methodology question:** AI-referral sessions are a small fraction of
total traffic in this kind of data (on the order of ~1% of sessions
elsewhere in the report). Given that, how many rows/sessions actually
support the "high-AI" comparison group? With a sparse denominator, a 9x
multiplier can be sensitive to a small number of high-traffic outlier
pages rather than a stable population-level pattern. Was a minimum-volume
threshold applied before computing this ratio (the way several other
findings in the report explicitly use volume filters), and does the 9x
figure hold up if recomputed on a different time window? This is the kind
of claim I'd want re-checked for stability before leaning on it heavily,
without doubting that AI referral traffic is worth watching.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Hugging Face Token Setup

To resolve the `KeyError: 'HF_TOKEN'`, please add your Hugging Face token to Colab's secrets manager. Follow these steps:

1.  Click the "🔑 Secrets" icon on the left sidebar.
2.  Click "+ New secret".
3.  Set the **Name** to `HF_TOKEN`.
4.  Paste your Hugging Face access token into the **Value** field.
5.  Ensure "Notebook access" is enabled for this secret.

After setting the secret, run the cell below to confirm it's accessible.

In [2]:
# Access the Hugging Face token from Colab's secrets manager
from google.colab import userdata
import os

# This will raise a KeyError if HF_TOKEN is not set in secrets or not enabled for notebook access.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

print("HF_TOKEN successfully loaded from secrets.")
print("You can now run the next code cell.")

HF_TOKEN successfully loaded from secrets.
You can now run the next code cell.


In [3]:
import duckdb, os
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
fact_april = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

# Rebuild the same features/label as w05_model.ipynb
features = con.sql(f"""
    SELECT f.content_hash_id, f.client_hash_id,
           SUM(f.gsc_impressions) as impressions_march,
           AVG(f.gsc_avg_position) as avg_position,
           DATEDIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days,
           SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) as days_with_ga4
    FROM {fact_march} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2, c.content_created_date
""").df()

april = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) as impressions_april FROM {fact_april} GROUP BY 1").df()
data = features.merge(april, on="content_hash_id", how="left").dropna()
data["is_declining"] = (data["impressions_april"] < data["impressions_march"]).astype(int)

X = data[["impressions_march", "avg_position", "content_age_days", "days_with_ga4"]].fillna(0)
y = data["is_declining"]
groups = data["client_hash_id"]

def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# BEFORE: naive random split — same client's pages can appear in both train and test
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(X_train_r, y_train_r)
p_random = precision_at_k(rf_random.predict_proba(X_test_r)[:,1], y_test_r, 50)

# AFTER: grouped split — no client's pages cross train/test (same as w05_model.ipynb)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(X_train_g, y_train_g)
p_grouped = precision_at_k(rf_grouped.predict_proba(X_test_g)[:,1], y_test_g, 50)

print(f"BEFORE (random split, client leakage possible)  Precision@50: {p_random:.3f}")
print(f"AFTER  (grouped split, no client crosses)        Precision@50: {p_grouped:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (random split, client leakage possible)  Precision@50: 0.900
AFTER  (grouped split, no client crosses)        Precision@50: 0.700


**Before/after — random split vs. grouped split:**

| Split type | Precision@50 |
|---|---:|
| BEFORE (random split, client leakage possible) | 0.900 |
| AFTER (grouped split, no client crosses) | 0.700 |

**What this shows:** the random split score (0.900) was noticeably
inflated compared to the honest, client-grouped estimate (0.700) — a gap
of 0.200, large enough to matter. This is consistent with client leakage:
when a random split is used, pages from the same client can appear in both
training and test sets, letting the model partly learn client-specific
patterns (a particular client's typical position range, content style, or
naming conventions) rather than signal that generalizes to clients it
hasn't seen. The grouped split — which was already used in w05_model.ipynb
— removes that shortcut, so 0.700 is the more honest number to report and
trust going forward.

This is a good illustration of why the split design is not a technical
detail to skip past: the same model, same features, same metric produced
two meaningfully different answers depending on one methodology choice.
Any claim about this model's performance should cite the grouped-split
number (0.700), not the random-split number (0.900), even though the
larger number is more flattering.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage audit (repeating the Week-3 hunt on my final feature set):**

- `impressions_march`, `avg_position`, `content_age_days`, `days_with_ga4`
  — all computed strictly from March 2026 data or a fixed past date
  (`content_created_date`). None touch April (the label window).
- No FlyRank product flags (`health_score`, `priority_score`, `action_type`,
  `refresh_tier`) used anywhere — confirmed by checking the exact column
  list pulled in every query across w03-w05.
- No `trend_direction` or `trend_pct` used as features (the exact label-
  derived trap flagged in the flyrank-data skill) — my label here is built
  independently from April impressions, not from any precomputed trend bucket.
- Train-test contamination check: no preprocessing (scaling, imputing) was
  fit on the full dataset before splitting — `fillna(0)` is a constant, not
  a fitted statistic, so it doesn't leak train/test information either way.
- One residual risk worth naming: `content_age_days` is derived from
  `content_created_date`, which is fixed and safe — but Section 2's error
  analysis in w05_model.ipynb flagged this feature as possibly encoding a
  spurious age-band pattern specific to the training split, not true leakage
  but a generalization risk worth watching.

**Confirms the split matters:** the 0.200-point gap between random and
grouped splits (Section 2) is itself indirect evidence that some
client-identifiable signal was being picked up under the random split —
reinforcing why `client_hash_id` is used only for grouping here, never as
a feature, and why the grouped split is the one reported in this notebook
and in w05_model.ipynb.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (from w05_model.ipynb):** "Both learned models clearly
beat both the rule and the base rate... the trained model clearly beat the
hand-written baseline."

**Rewritten in safe language:** On this test split, the random forest's
Precision@50 (0.700) was observed to exceed both the baseline rule (0.500)
and the base rate (0.578); logistic regression showed a similar directional
pattern (0.660). This is decision-support evidence for this specific split
and time window, not a guarantee of future performance — the honest-split
comparison in Section 2 of this notebook is a more conservative estimate,
and either should be re-validated on new data before being treated as a
stable result.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.